##### Short cohort: modulated VBM

In [5]:
import os
import re
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import ants
import nibabel as nib
import numpy as np
import pandas as pd
from tqdm import tqdm

os.environ.setdefault("FSLDIR", "/Users/william.wakefield/fsl")
_fsl_bin = f"{os.environ['FSLDIR']}/share/fsl/bin"
if _fsl_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _fsl_bin + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("FSLOUTPUTTYPE", "NIFTI_GZ")

'NIFTI_GZ'

In [6]:
SHORT_T1 = Path("model_data/adni/t1_short_data")
SHORT_DTI = Path("model_data/adni/dti_short_data")
T1_DCM = SHORT_T1 / "t1_mri_dcm"
DTI_DCM = SHORT_DTI / "dti_dcm"
T1_RAW = SHORT_T1 / "t1_mri_raw"
DTI_RAW = SHORT_DTI / "dti_raw"
T1_VBM = SHORT_T1 / "t1_short_modulated_vbm"
DTI_VBM = SHORT_DTI / "dti_short_modulated_vbm"

T1_STRIP = T1_VBM / "t1_short_skull_strip"
T1_SEG = T1_VBM / "t1_short_seg_native"
T1_WARP = T1_VBM / "t1_short_vbm_warps"
T1_JAC = T1_VBM / "t1_short_jacobian"
T1_PVE_MNI = T1_VBM / "t1_short_pve_mni"
T1_PVE_MOD = T1_VBM / "t1_short_pve_mni_modulated"
T1_PVE_SMOOTH = T1_VBM / "t1_short_pve_mni_smoothed"

DTI_STRIP = DTI_VBM / "dti_short_skull_strip"
DTI_FIT = DTI_VBM / "dti_short_dtifit"
ADC_T1 = DTI_VBM / "dti_short_reg_t1_native"
ADC_MNI = DTI_VBM / "dti_short_mni"
ADC_SMOOTH = DTI_VBM / "dti_short_mni_smoothed"

MNI_T1 = Path("model_data/mni_t1.nii")

for d in (
    T1_RAW, DTI_RAW, T1_STRIP, T1_SEG, T1_WARP, T1_JAC,
    T1_PVE_MNI, T1_PVE_MOD, T1_PVE_SMOOTH,
    DTI_STRIP, DTI_FIT, ADC_T1, ADC_MNI, ADC_SMOOTH,
):
    d.mkdir(parents=True, exist_ok=True)

_IMAGE_DIR = re.compile(r"^I\d+$\Z")

In [7]:
def short_image_scan_dirs(dcm_root: Path) -> list[Path]:
    """Folders named I<digits> (ADNI image UID) that contain DICOMs.
    Typical layout: .../PTID/SeriesName/DateTime/I<id>/*.dcm — depth varies, so we rglob.
    """
    if not dcm_root.is_dir():
        return []
    dirs: list[Path] = []
    for p in sorted(dcm_root.rglob("*")):
        if not p.is_dir() or not _IMAGE_DIR.fullmatch(p.name):
            continue
        if any(p.glob("*.dcm")) or any(p.glob("*.DCM")):
            dirs.append(p)
    return dirs


def subject_id_from_image_dir(image_dir: Path) -> str:
    return image_dir.parent.parent.parent.name


print("FSLOUTPUTTYPE:", os.environ.get("FSLOUTPUTTYPE"))

FSLOUTPUTTYPE: NIFTI_GZ


In [7]:
# --- DICOM → NIfTI (T1): dcm2niix ----------------------------------------------
scan_dirs = short_image_scan_dirs(T1_DCM)
print(f"Found {len(scan_dirs)} T1 image folders")

failed = []
skipped = 0

for scan_dir in tqdm(scan_dirs, desc="dcm2niix T1"):
    subj = subject_id_from_image_dir(scan_dir)
    out_name = f"{scan_dir.name}_{subj}"
    if (T1_RAW / f"{out_name}.nii").exists():
        skipped += 1
        continue
    r = subprocess.run(
        ["dcm2niix", "-z", "n", "-f", out_name, "-o", str(T1_RAW), "-b", "n", str(scan_dir)],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        failed.append((out_name, r.stderr.strip()))

nii = list(T1_RAW.glob("I*.nii"))
print(f"T1 NIfTI: {len(nii)} files ({skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

Found 295 T1 image folders


dcm2niix T1: 100%|██████████| 295/295 [00:59<00:00,  4.95it/s]

T1 NIfTI: 295 files (0 skipped, 1 failed)
  I820315_016_S_4952: 


In [7]:
# --- DICOM → NIfTI (DTI): dcm2niix with bvec/bval ----------------------------
scan_dirs = short_image_scan_dirs(DTI_DCM)
print(f"Found {len(scan_dirs)} DTI image folders")

failed = []
skipped = 0

for scan_dir in tqdm(scan_dirs, desc="dcm2niix DTI"):
    subj = subject_id_from_image_dir(scan_dir)
    out_name = f"{scan_dir.name}_{subj}"
    if (DTI_RAW / f"{out_name}.nii").exists():
        skipped += 1
        continue
    r = subprocess.run(
        ["dcm2niix", "-z", "n", "-f", out_name, "-o", str(DTI_RAW), "-b", "y", str(scan_dir)],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        failed.append((out_name, r.stderr.strip()))

nii = list(DTI_RAW.glob("I*.nii"))
print(f"DTI NIfTI: {len(nii)} files ({skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

Found 142 DTI image folders


dcm2niix DTI: 100%|██████████| 142/142 [01:34<00:00,  1.50it/s]

DTI NIfTI: 142 files (0 skipped, 0 failed)


#### DTI bval/bvec QC — validate all 5 DICOM format edge cases before dtifit

Fixes applied here:
- **GE b=0 encoded as -1**: clamp to 0 in-place in `.bval` files
- **Siemens MOSAIC**: verify dcm2niix read CSA `(0019,100C)` — flag if all bvals are 0
- **Philips enhanced (1 DCM / 2880 frames)**: verify volume count is correct
- **Insufficient directions** (e.g. `141_S_6116`, 4 dirs): mark for exclusion — dtifit produces degenerate tensors below 6 non-collinear directions

In [4]:
# --- DTI bval/bvec QC: fix GE -1 encoding, detect bad fits, build exclusion set ---
MIN_DIRS = 6   # dtifit hard minimum

issues = []          # (stem, issue_type, detail)
dti_exclude = set()  # stems to skip in BET + dtifit

primary_pat_bval = re.compile(r"^I\d+_\d{3}_S_\d+\.bval$")
bval_files = sorted(f for f in DTI_RAW.glob("*.bval") if primary_pat_bval.match(f.name))
print(f"Validating {len(bval_files)} bval files ...")

for bval_path in bval_files:
    stem = bval_path.stem
    bvec_path = bval_path.with_suffix(".bvec")
    nii_path  = DTI_RAW / f"{stem}.nii"

    # 1. Load bvals
    try:
        bvals = np.loadtxt(str(bval_path)).flatten()
    except Exception as e:
        issues.append((stem, "bval_read_error", str(e)))
        dti_exclude.add(stem)
        continue

    # 2. Fix GE b=0 encoded as -1 in standard DICOM tag (0018,9087).
    #    dcm2niix reads (0043,1039) instead and usually gets 0, but write
    #    the corrected file defensively in case an older dcm2niix is used.
    if np.any(bvals == -1):
        bvals_fixed = np.where(bvals == -1, 0.0, bvals)
        np.savetxt(str(bval_path), bvals_fixed[np.newaxis, :], fmt="%.4f")
        issues.append((stem, "GE_neg1_bval_fixed",
                        f"{int((bvals == -1).sum())} values clamped -1→0"))
        bvals = bvals_fixed

    # 3. Detect Siemens CSA bval failure: if dcm2niix could not read (0019,100C)
    #    every volume gets bval=0 — catches complete CSA read failure.
    n_b0  = int((bvals <  100).sum())
    n_dwi = int((bvals >= 100).sum())
    if n_dwi == 0:
        issues.append((stem, "all_bvals_zero",
                        f"{len(bvals)} vols, 0 DWI dirs — possible Siemens CSA read failure"))
        dti_exclude.add(stem)
        continue

    # 4. Insufficient gradient directions — dtifit degenerates silently below 6.
    if n_dwi < MIN_DIRS:
        issues.append((stem, "insufficient_dirs",
                        f"{n_dwi} directions (need ≥{MIN_DIRS}); subject has too few DWI volumes"))
        dti_exclude.add(stem)
        continue

    # 5. bvec column count must match bval count.
    if not bvec_path.exists():
        issues.append((stem, "missing_bvec", ""))
        dti_exclude.add(stem)
        continue
    try:
        bvecs = np.loadtxt(str(bvec_path))
        n_vecs = bvecs.shape[1] if bvecs.ndim == 2 else len(bvecs)
        if n_vecs != len(bvals):
            issues.append((stem, "bvec_bval_mismatch",
                            f"bval={len(bvals)} bvec cols={n_vecs}"))
            dti_exclude.add(stem)
            continue
    except Exception as e:
        issues.append((stem, "bvec_read_error", str(e)))
        dti_exclude.add(stem)
        continue

    # 6. NIfTI 4th dimension must match bval count (catches Philips enhanced
    #    where dcm2niix only output 1 frame instead of all frames).
    if nii_path.exists():
        try:
            shape = nib.load(str(nii_path)).shape
            n_vols = shape[3] if len(shape) == 4 else 1
            if n_vols != len(bvals):
                issues.append((stem, "vol_bval_mismatch",
                                f"nifti vols={n_vols} bval count={len(bvals)}"))
                dti_exclude.add(stem)
        except Exception as e:
            issues.append((stem, "nifti_read_error", str(e)))

# 7. Philips enhanced: dcm2niix sometimes emits auxiliary echo files (_e1, _e2).
#    These are not the main DWI volume; flag them so the operator knows.
for extra in sorted(DTI_RAW.glob("I*_*_S_*_e*.nii")):
    base = re.sub(r"_e\d+.*", "", extra.stem)
    issues.append((base, "philips_extra_echo_file", extra.name))

print(f"QC complete: {len(bval_files)} series checked")
print(f"  Excluded (unfittable): {len(dti_exclude)}")
for s in sorted(dti_exclude):
    itype  = next((t for st, t, _ in issues if st == s), "?")
    detail = next((d for st, _, d in issues if st == s), "?")
    print(f"    EXCLUDE {s}: {itype} — {detail}")
non_fatal = [(s, t, d) for s, t, d in issues if s not in dti_exclude]
if non_fatal:
    print(f"  Non-fatal warnings ({len(non_fatal)}):")
    for s, t, d in non_fatal:
        print(f"    WARN    {s}: {t} — {d}")

Validating 142 bval files ...
QC complete: 142 series checked
  Excluded (unfittable): 3
    EXCLUDE I1181934_153_S_6755: all_bvals_zero — 55 vols, 0 DWI dirs — possible Siemens CSA read failure
    EXCLUDE I1581804_114_S_6057: all_bvals_zero — 31 vols, 0 DWI dirs — possible Siemens CSA read failure
    EXCLUDE I923015_114_S_6057: all_bvals_zero — 31 vols, 0 DWI dirs — possible Siemens CSA read failure


In [ ]:
# --- BET (DTI): extract b0, skull-strip, apply mask --------------------------
# Uses the actual first b=0 volume index from bvals (not always volume 0 —
# Philips classic and GE sometimes place b0 mid-series).
# Subjects in dti_exclude (< 6 dirs, all-bvals-zero, bvec mismatch) are skipped.
primary_pat = re.compile(r"^I\d+_\d{3}_S_\d+\.nii$")
dti_raw_files = sorted(f for f in DTI_RAW.glob("*.nii") if primary_pat.match(f.name))

if "dti_exclude" not in dir():
    dti_exclude = set()
    print("WARNING: dti_exclude not in scope — run the QC cell above to exclude bad subjects")

dti_raw_files = [f for f in dti_raw_files if f.stem not in dti_exclude]
print(f"Skull-strip {len(dti_raw_files)} DTI volumes "
      f"(bet -m -f 0.3, {len(dti_exclude)} excluded by QC)")

failed = []
skipped = 0

for dti_path in tqdm(dti_raw_files, desc="BET DTI"):
    stem = dti_path.stem
    out_nii   = DTI_STRIP / f"{stem}.nii.gz"
    mask_path = DTI_STRIP / f"{stem}_mask.nii.gz"
    bval_dst  = DTI_STRIP / f"{stem}.bval"
    bvec_dst  = DTI_STRIP / f"{stem}.bvec"

    if out_nii.exists() and mask_path.exists() and bval_dst.exists() and bvec_dst.exists():
        skipped += 1
        continue

    # Find the first b=0 index — fslroi extracts that specific volume.
    bval_src = DTI_RAW / f"{stem}.bval"
    bvec_src = DTI_RAW / f"{stem}.bvec"
    b0_idx = 0
    if bval_src.exists():
        try:
            _bv = np.loadtxt(str(bval_src)).flatten()
            _idxs = np.where(_bv < 100)[0]
            b0_idx = int(_idxs[0]) if len(_idxs) > 0 else 0
        except Exception:
            b0_idx = 0

    b0_path       = DTI_STRIP / f"{stem}_b0.nii.gz"
    b0_brain_base = DTI_STRIP / f"{stem}_b0_brain"
    b0_brain      = DTI_STRIP / f"{stem}_b0_brain.nii.gz"
    b0_mask       = DTI_STRIP / f"{stem}_b0_brain_mask.nii.gz"

    try:
        subprocess.run(
            ["fslroi", str(dti_path), str(b0_path), str(b0_idx), "1"],
            capture_output=True, text=True, check=True,
        )
        subprocess.run(
            ["bet", str(b0_path), str(b0_brain_base), "-m", "-f", "0.3"],
            capture_output=True, text=True, check=True,
        )
        subprocess.run(
            ["fslmaths", str(dti_path), "-mas", str(b0_mask), str(out_nii)],
            capture_output=True, text=True, check=True,
        )
        shutil.move(str(b0_mask), str(mask_path))

        if bval_src.exists():
            shutil.copy(bval_src, bval_dst)
        if bvec_src.exists():
            shutil.copy(bvec_src, bvec_dst)

        for tmp in [b0_path, b0_brain]:
            if tmp.exists():
                tmp.unlink()

    except subprocess.CalledProcessError as e:
        failed.append((stem, e.stderr.strip() if e.stderr else str(e)))
    except Exception as e:
        failed.append((stem, str(e)))

stripped = list(DTI_STRIP.glob("I*_[0-9]*_S_[0-9]*.nii.gz"))
masks    = list(DTI_STRIP.glob("*_mask.nii.gz"))
bvals_l  = list(DTI_STRIP.glob("*.bval"))
bvecs_l  = list(DTI_STRIP.glob("*.bvec"))
print(f"DTI skull stripping complete:")
print(f"  {len(stripped)} stripped | {len(masks)} masks | {len(bvals_l)} bval | {len(bvecs_l)} bvec")
print(f"  ({skipped} skipped, {len(dti_exclude)} excluded by QC, {len(failed)} failed)")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

In [ ]:
# --- dtifit: fit tensor model → FA, MD, etc. ----------------------------------
# dti_exclude subjects are skipped — they have degenerate bvals/bvecs that
# would produce near-zero or nonsense FA/MD maps (silent failure in FSL dtifit).
dtifit_primary = re.compile(r"^I\d+_\d{3}_S_\d+\.nii\.gz$")
dti_strip_files = sorted(f for f in DTI_STRIP.glob("*.nii.gz") if dtifit_primary.match(f.name))

if "dti_exclude" not in dir():
    dti_exclude = set()
    print("WARNING: dti_exclude not in scope — run the QC cell to exclude bad subjects")

dti_strip_files = [f for f in dti_strip_files if f.stem not in dti_exclude]
print(f"dtifit on {len(dti_strip_files)} skull-stripped DTI volumes "
      f"({len(dti_exclude)} excluded by QC)")

failed = []
skipped = 0

for dti_path in tqdm(dti_strip_files, desc="dtifit"):
    stem = dti_path.name.replace(".nii.gz", "")
    mask_path = DTI_STRIP / f"{stem}_mask.nii.gz"
    bval_path = DTI_STRIP / f"{stem}.bval"
    bvec_path = DTI_STRIP / f"{stem}.bvec"

    subj_dir = DTI_FIT / stem
    subj_dir.mkdir(parents=True, exist_ok=True)
    out_base = subj_dir / stem
    fa_file  = subj_dir / f"{stem}_FA.nii.gz"

    if fa_file.exists():
        skipped += 1
        continue

    missing = [p for p in [mask_path, bval_path, bvec_path] if not p.exists()]
    if missing:
        failed.append((stem, f"missing inputs: {[p.name for p in missing]}"))
        continue

    r = subprocess.run(
        [
            "dtifit",
            "-k", str(dti_path),
            "-m", str(mask_path),
            "-r", str(bvec_path),
            "-b", str(bval_path),
            "-o", str(out_base),
        ],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        failed.append((stem, r.stderr.strip()))

fa_maps = list(DTI_FIT.glob("*/*_FA.nii.gz"))
md_maps = list(DTI_FIT.glob("*/*_MD.nii.gz"))
print(f"dtifit complete: {len(fa_maps)} FA | {len(md_maps)} MD | "
      f"{skipped} skipped | {len(dti_exclude)} excluded | {len(failed)} failed")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

In [4]:
# --- BET (native T1) ------------------------------------------------------------
raw_files = sorted(T1_RAW.glob("I*.nii"))
print(f"Skull-strip {len(raw_files)} T1 volumes (bet -R -f 0.4)")


def _run_bet(raw_path: Path, out_nii: Path):
    r = subprocess.run(
        ["bet", str(raw_path), str(out_nii), "-R", "-f", "0.4"],
        capture_output=True, text=True,
    )
    return raw_path.stem, r.returncode, r.stderr.strip()


failed = []
skipped = 0
tasks = []
for raw_path in raw_files:
    out_nii = T1_STRIP / f"{raw_path.stem}.nii.gz"
    if out_nii.exists():
        skipped += 1
        continue
    tasks.append((raw_path, out_nii))

bet_workers = int(os.environ["BET_MAX_WORKERS"]) if os.environ.get("BET_MAX_WORKERS") else (os.cpu_count() or 8)
max_workers = max(1, min(len(tasks) or 1, bet_workers))
print(f"BET: {len(tasks)} jobs, {max_workers} workers ({skipped} already done)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_run_bet, a, b) for a, b in tasks]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="BET"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

print(f"Stripped: {len(list(T1_STRIP.glob('*.nii.gz')))} | failed: {len(failed)}")

Skull-strip 295 T1 volumes (bet -R -f 0.4)
BET: 295 jobs, 16 workers (0 already done)


BET: 100%|██████████| 295/295 [10:56<00:00,  2.22s/it]

Stripped: 295 | failed: 0


In [5]:
# --- FAST (native T1) -----------------------------------------------------------
# Speed: use all CPU cores by default (was: cpu_count − 2). Cap with FAST_MAX_WORKERS.
# Extra speed (rougher PVEs): export FAST_QUICK=1  →  -I 2 -O 2 -W 8
# Or tune: FAST_I / FAST_O / FAST_W (FSL defaults: 4 / 4 / 15).
# BET also writes companion files (e.g. *_mask.nii.gz). Only run FAST on the
# brain-extracted volume that matches each raw stem: {stem}.nii.gz — not masks.
_brain_paths = [T1_STRIP / f"{raw_path.stem}.nii.gz" for raw_path in sorted(T1_RAW.glob("I*.nii"))]
stripped = [p for p in _brain_paths if p.exists()]
_n_missing = len(_brain_paths) - len(stripped)
print(
    f"FAST on {len(stripped)} brain-extracted T1s "
    f"({_n_missing} without strip yet; skipped *_mask.nii.gz and other BET sidecars)"
)

if os.environ.get("FAST_QUICK", "").lower() in ("1", "true", "yes"):
    _I, _O, _W = "2", "2", "8"
else:
    _I = os.environ.get("FAST_I", "4")
    _O = os.environ.get("FAST_O", "4")
    _W = os.environ.get("FAST_W", "15")


def _run_fast(strip_path: Path, out_base: Path):
    r = subprocess.run(
        [
            "fast", "-t", "1", "-n", "3", "-B", "-b",
            "-I", _I, "-O", _O, "-W", _W,
            "-o", str(out_base), str(strip_path),
        ],
        capture_output=True,
        text=True,
    )
    stem = strip_path.name.replace(".nii.gz", "")
    return stem, r.returncode, r.stderr.strip()


failed = []
skipped = 0
tasks = []
for strip_path in stripped:
    stem = strip_path.name.replace(".nii.gz", "")
    subj_dir = T1_SEG / stem
    subj_dir.mkdir(parents=True, exist_ok=True)
    out_base = subj_dir / stem
    pve = [subj_dir / f"{stem}_pve_{i}.nii.gz" for i in range(3)]
    if all(p.exists() for p in pve):
        skipped += 1
        continue
    tasks.append((strip_path, out_base))

_n = int(os.environ["FAST_MAX_WORKERS"]) if os.environ.get("FAST_MAX_WORKERS") else (os.cpu_count() or 8)
max_workers = max(1, min(len(tasks) or 1, _n))
print(
    f"FAST: {len(tasks)} jobs ({skipped} skipped) | workers={max_workers} | "
    f"-I {_I} -O {_O} -W {_W}"
)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_run_fast, a, b) for a, b in tasks]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="FAST"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

csf = list(T1_SEG.glob("*/*_pve_0.nii.gz"))
print(f"PVE maps: {len(csf)} subjects | FAST failed: {len(failed)}")

FAST on 295 brain-extracted T1s (0 without strip yet; skipped *_mask.nii.gz and other BET sidecars)
FAST: 295 jobs (0 skipped) | workers=16 | -I 4 -O 4 -W 15


FAST: 100%|██████████| 295/295 [2:12:14<00:00, 26.90s/it]  

PVE maps: 295 subjects | FAST failed: 0


In [6]:
# --- SyN: raw T1 → MNI (forward warps for VBM) --------------------------------
mni_t1 = ants.image_read(str(MNI_T1))
raw_files = sorted(T1_RAW.glob("I*.nii"))
print(f"Register {len(raw_files)} native T1 → MNI")

failed = []
skipped = 0
for raw_path in tqdm(raw_files, desc="SyN T1→MNI"):
    stem = raw_path.stem
    out_warped = T1_WARP / f"{stem}_warpedT1.nii.gz"
    out_affine = T1_WARP / f"{stem}_0GenericAffine.mat"
    out_warp = T1_WARP / f"{stem}_1Warp.nii.gz"
    out_inv = T1_WARP / f"{stem}_1InverseWarp.nii.gz"
    if out_warped.exists() and out_affine.exists() and out_warp.exists() and out_inv.exists():
        skipped += 1
        continue
    try:
        moving = ants.image_read(str(raw_path))
        reg = ants.registration(fixed=mni_t1, moving=moving, type_of_transform="SyN")
        ants.image_write(reg["warpedmovout"], str(out_warped))
        for tx_path in reg["fwdtransforms"]:
            tx = Path(tx_path)
            if "GenericAffine" in tx.name:
                shutil.copy(tx, out_affine)
            elif "Warp" in tx.name and "Inverse" not in tx.name:
                shutil.copy(tx, out_warp)
        for tx_path in reg["invtransforms"]:
            tx = Path(tx_path)
            if "InverseWarp" in tx.name:
                shutil.copy(tx, out_inv)
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Warps: {len(list(T1_WARP.glob('*_1Warp.nii.gz')))} | skipped {skipped} | failed {len(failed)}")

Register 295 native T1 → MNI


SyN T1→MNI: 100%|██████████| 295/295 [39:03<00:00,  7.94s/it]

Warps: 295 | skipped 0 | failed 0


In [7]:
# --- Jacobian det (from SyN warp + affine) ------------------------------------
warp_files = sorted(T1_WARP.glob("*_1Warp.nii.gz"))
print(f"Jacobians for {len(warp_files)} warps")

failed = []
skipped = 0

for warp_path in tqdm(warp_files, desc="Jacobian"):
    stem = warp_path.name.replace("_1Warp.nii.gz", "")
    affine_path = T1_WARP / f"{stem}_0GenericAffine.mat"
    out_jac = T1_JAC / f"{stem}_jacobian.nii.gz"
    if out_jac.exists():
        skipped += 1
        continue
    if not affine_path.exists():
        failed.append((stem, "missing affine"))
        continue
    try:
        jac_warp = ants.create_jacobian_determinant_image(
            domain_image=mni_t1, tx=str(warp_path), do_log=False, geom=False,
        )
        affine_tx = ants.read_transform(str(affine_path))
        params = np.asarray(affine_tx.parameters)
        det_affine = abs(float(np.linalg.det(params[:9].reshape(3, 3))))
        jac_total = jac_warp * det_affine
        ants.image_write(jac_total, str(out_jac))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Jacobian maps: {len(list(T1_JAC.glob('*_jacobian.nii.gz')))} | failed {len(failed)}")

Jacobians for 295 warps


Jacobian: 100%|██████████| 295/295 [01:28<00:00,  3.33it/s]

Jacobian maps: 295 | failed 0


In [8]:
# --- Warp native PVEs → MNI ----------------------------------------------------
subject_dirs = sorted(p for p in T1_SEG.iterdir() if p.is_dir())
print(f"Warp PVEs for {len(subject_dirs)} subjects")

failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="PVE→MNI"):
    stem = subj_dir.name
    out_sub = T1_PVE_MNI / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    affine = T1_WARP / f"{stem}_0GenericAffine.mat"
    warp = T1_WARP / f"{stem}_1Warp.nii.gz"
    if not affine.exists() or not warp.exists():
        failed.append((stem, "missing transforms"))
        continue
    tx = [str(warp), str(affine)]
    if all((out_sub / f"{stem}_pve_{i}_mni.nii.gz").exists() for i in range(3)):
        skipped += 1
        continue
    try:
        for i in range(3):
            in_pve = subj_dir / f"{stem}_pve_{i}.nii.gz"
            out_pve = out_sub / f"{stem}_pve_{i}_mni.nii.gz"
            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}"))
                break
            moving = ants.image_read(str(in_pve))
            warped = ants.apply_transforms(
                fixed=mni_t1, moving=moving, transformlist=tx, interpolator="linear",
            )
            ants.image_write(warped, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"MNI PVE dirs: {len(list(T1_PVE_MNI.iterdir()))} | failed {len(failed)}")

Warp PVEs for 295 subjects


PVE→MNI: 100%|██████████| 295/295 [04:30<00:00,  1.09it/s]

MNI PVE dirs: 296 | failed 0


In [9]:
# --- Modulate MNI PVE × Jacobian ------------------------------------------------
subject_dirs = sorted(p for p in T1_PVE_MNI.iterdir() if p.is_dir())
print(f"Modulate {len(subject_dirs)} subjects")

failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Modulate"):
    stem = subj_dir.name
    out_sub = T1_PVE_MOD / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    jac_path = T1_JAC / f"{stem}_jacobian.nii.gz"
    if not jac_path.exists():
        failed.append((stem, "missing jacobian"))
        continue
    if all((out_sub / f"{stem}_pve_{i}_mod.nii.gz").exists() for i in range(3)):
        skipped += 1
        continue
    try:
        jac_img = ants.image_read(str(jac_path))
        jac_arr = jac_img.numpy()
        for i in range(3):
            in_pve = subj_dir / f"{stem}_pve_{i}_mni.nii.gz"
            out_pve = out_sub / f"{stem}_pve_{i}_mod.nii.gz"
            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}_mni"))
                continue
            pve_img = ants.image_read(str(in_pve))
            mod_arr = pve_img.numpy() * jac_arr
            mod_img = ants.from_numpy(
                mod_arr, origin=pve_img.origin,
                spacing=pve_img.spacing, direction=pve_img.direction,
            )
            ants.image_write(mod_img, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Modulated: {len(list(T1_PVE_MOD.glob('*/*_pve_0_mod.nii.gz')))} CSF-like | skipped {skipped} | failures {len(failed)}")

Modulate 295 subjects


Modulate: 100%|██████████| 295/295 [01:12<00:00,  4.10it/s]

Modulated: 295 CSF-like | skipped 0 | failures 0


In [ ]:
# --- Smooth modulated PVE (2.5 mm FWHM) -----------------------------------------
fwhm_mm = 2.5
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
subject_dirs = sorted(p for p in T1_PVE_MOD.iterdir() if p.is_dir())
print(f"Smooth {len(subject_dirs)} subjects @ {fwhm_mm} mm FWHM")

failed = []
skipped = 0

for subj_dir in tqdm(subject_dirs, desc="Smooth"):
    stem = subj_dir.name
    out_sub = T1_PVE_SMOOTH / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    if all((out_sub / f"{stem}_pve_{i}_mod_s2p5.nii.gz").exists() for i in range(3)):
        skipped += 1
        continue
    try:
        for i in range(3):
            in_pve = subj_dir / f"{stem}_pve_{i}_mod.nii.gz"
            out_pve = out_sub / f"{stem}_pve_{i}_mod_s2p5.nii.gz"
            if out_pve.exists():
                continue
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}_mod"))
                continue
            img = ants.image_read(str(in_pve))
            sm = ants.smooth_image(
                img, sigma=sigma_mm, sigma_in_physical_coordinates=True,
            )
            ants.image_write(sm, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(
    "Smoothed:", len(list(T1_PVE_SMOOTH.glob("*/*_pve_1_mod_s2p5.nii.gz"))),
    "GM | failures:", len(failed),
)

Smooth 295 subjects @ 2.5 mm FWHM


Smooth: 100%|██████████| 295/295 [01:28<00:00,  3.33it/s]

Smoothed: 295 GM | failures: 0


In [3]:
# --- Map each MD volume → matching skull-stripped T1 (by closest scan date) ---
import re
import pandas as pd
from pathlib import Path

subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")
_date_pat = re.compile(r"^(\d{4}-\d{2}-\d{2})")
_img_pat  = re.compile(r"^I?(\d+)$")

def _build_date_map(dcm_root: Path) -> dict:
    mapping = {}
    for subj_dir in dcm_root.iterdir():
        if not subj_dir.is_dir(): continue
        for mod_dir in subj_dir.iterdir():
            if not mod_dir.is_dir(): continue
            for date_dir in mod_dir.iterdir():
                if not date_dir.is_dir(): continue
                m = _date_pat.match(date_dir.name)
                if not m: continue
                scan_date = pd.Timestamp(m.group(1))
                for img_dir in date_dir.iterdir():
                    if not img_dir.is_dir(): continue
                    im = _img_pat.match(img_dir.name)
                    if im:
                        mapping[im.group(1)] = scan_date
    return mapping

def _stem_date(stem: str, date_map: dict):
    img_id = stem.lstrip("I").split("_")[0]
    return date_map.get(img_id)

t1_date_raw  = _build_date_map(T1_DCM)
dti_date_raw = _build_date_map(DTI_DCM)
print(f"T1 date map: {len(t1_date_raw)} | DTI date map: {len(dti_date_raw)}")

t1_by_subj: dict[str, list[str]] = {}
for f in sorted(T1_STRIP.glob("*.nii.gz")):
    stem = f.name.replace(".nii.gz", "")
    m = subj_pat.match(stem)
    if m:
        t1_by_subj.setdefault(m.group(1), []).append(stem)

adc_by_subj: dict[str, list[tuple[str, Path]]] = {}
adc_paths = sorted(DTI_FIT.glob("*/*_MD.nii.gz"))
for p in adc_paths:
    stem = p.parent.name
    m = subj_pat.match(stem)
    if m:
        adc_by_subj.setdefault(m.group(1), []).append((stem, p))

dti_to_t1: dict[str, str] = {}
for subj, adc_list in adc_by_subj.items():
    t1_list = sorted(t1_by_subj.get(subj, []))
    if not t1_list:
        continue
    for adc_stem, _ in sorted(adc_list, key=lambda x: x[0]):
        dti_dt = _stem_date(adc_stem, dti_date_raw)
        if dti_dt is None:
            # fallback: sort-order pairing
            idx = sorted(adc_list, key=lambda x: x[0]).index((adc_stem, _))
            dti_to_t1[adc_stem] = t1_list[min(idx, len(t1_list) - 1)]
            continue
        t1_with_dates = [(t, _stem_date(t, t1_date_raw)) for t in t1_list]
        t1_with_dates = [(t, d) for t, d in t1_with_dates if d is not None]
        if not t1_with_dates:
            dti_to_t1[adc_stem] = t1_list[0]
        else:
            dti_to_t1[adc_stem] = min(t1_with_dates, key=lambda x: abs(x[1] - dti_dt))[0]

print(f"MD files: {len(adc_paths)} | matched to T1: {len(dti_to_t1)}")

T1 date map: 295 | DTI date map: 142
MD files: 142 | matched to T1: 142


In [ ]:
def _rigid_md_to_t1(md_path: Path):
    adc_stem = md_path.parent.name
    dti_stem = adc_stem
    t1_stem = dti_to_t1.get(dti_stem)
    fa_path = md_path.parent / f"{adc_stem}_FA.nii.gz"

    if t1_stem is None:
        return dti_stem, "fail", "no matching T1 for subject"

    out_fa  = ADC_T1 / f"{dti_stem}_FA.nii.gz"
    out_md  = ADC_T1 / f"{dti_stem}_MD.nii.gz"
    out_tx  = ADC_T1 / f"{dti_stem}_rigid.mat"

    if out_md.exists() and out_tx.exists():
        return dti_stem, "skip", ""

    t1_path = T1_STRIP / f"{t1_stem}.nii.gz"
    if not t1_path.exists():
        return dti_stem, "fail", f"missing T1: {t1_path.name}"
    for p in [fa_path, md_path]:
        if not p.exists():
            return dti_stem, "fail", f"missing {p.name}"

    try:
        fixed = ants.image_read(str(t1_path))

        # Reorient FA and MD to canonical RAS before registration.
        # Some acquisitions are stored in LAS, causing ~200 mm origin mismatches
        # that defeat identity-initialized rigid registration.
        import nibabel as _nib, tempfile as _tmp, os as _os

        def _to_ras_ants(path):
            img = _nib.load(str(path))
            img_ras = _nib.as_closest_canonical(img)
            with _tmp.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
                tmp = f.name
            _nib.save(img_ras, tmp)
            out = ants.image_read(tmp)
            _os.unlink(tmp)
            return out

        moving_fa = _to_ras_ants(fa_path)
        moving_md = _to_ras_ants(md_path)

        reg = ants.registration(
            fixed=fixed,
            moving=moving_fa,
            type_of_transform="Rigid",
            initial_transform="identity",
            aff_metric="mattes",
        )

        ants.image_write(reg["warpedmovout"], str(out_fa))

        tx_saved = False
        for tx in reg["fwdtransforms"]:
            tx_p = Path(tx)
            if tx_p.suffix == ".mat" or "GenericAffine" in tx_p.name:
                shutil.copy(tx, out_tx)
                tx_saved = True
                break
        if not tx_saved:
            return dti_stem, "fail", "no .mat in fwdtransforms"

        md_in_t1 = ants.apply_transforms(
            fixed=fixed, moving=moving_md,
            transformlist=reg["fwdtransforms"],
            interpolator="linear",
        )
        ants.image_write(md_in_t1, str(out_md))
        return dti_stem, "ok", ""

    except Exception as e:
        return dti_stem, "fail", str(e)

In [5]:
# --- Warp MD (T1 space) → MNI using same SyN warps as T1 ---------------------
# Rebuild dti_to_t1 from disk in case kernel was restarted after the rigid step.
if "dti_to_t1" not in dir() or not dti_to_t1:
    _subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")
    _t1_by_subj: dict[str, list[str]] = {}
    for f in sorted(T1_STRIP.glob("*.nii.gz")):
        stem = f.name.replace(".nii.gz", "")
        m = _subj_pat.match(stem)
        if m:
            _t1_by_subj.setdefault(m.group(1), []).append(stem)
    _adc_by_subj: dict[str, list[str]] = {}
    for p in sorted(ADC_T1.glob("*_MD.nii.gz")):
        stem = p.name.replace("_MD.nii.gz", "")
        m = _subj_pat.match(stem)
        if m:
            _adc_by_subj.setdefault(m.group(1), []).append(stem)
    dti_to_t1 = {}
    for subj, adc_list in _adc_by_subj.items():
        t1_list = sorted(_t1_by_subj.get(subj, []))
        for i, adc_stem in enumerate(sorted(adc_list)):
            t1_stem = t1_list[i] if i < len(t1_list) else t1_list[-1] if t1_list else None
            if t1_stem:
                dti_to_t1[adc_stem] = t1_stem
    print(f"Rebuilt dti_to_t1: {len(dti_to_t1)} entries")

if "mni_t1" not in dir():
    mni_t1 = ants.image_read(str(MNI_T1))

md_t1_files = sorted(ADC_T1.glob("*_MD.nii.gz"))
print(f"Apply T1 warps to {len(md_t1_files)} MD maps")


def _warp_md_to_mni(md_t1_path: Path):
    adc_stem = md_t1_path.name.replace("_MD.nii.gz", "")
    t1_stem = dti_to_t1.get(adc_stem)
    out_md = ADC_MNI / f"{adc_stem}_MD.nii.gz"
    if out_md.exists():
        return adc_stem, "skip", ""
    if t1_stem is None:
        return adc_stem, "fail", "no T1 map"
    affine = T1_WARP / f"{t1_stem}_0GenericAffine.mat"
    warp = T1_WARP / f"{t1_stem}_1Warp.nii.gz"
    if not affine.exists() or not warp.exists():
        return adc_stem, "fail", "missing warps"
    try:
        moving = ants.image_read(str(md_t1_path))
        warped = ants.apply_transforms(
            fixed=mni_t1,
            moving=moving,
            transformlist=[str(warp), str(affine)],
            interpolator="linear",
        )
        ants.image_write(warped, str(out_md))
        return adc_stem, "ok", ""
    except Exception as e:
        return adc_stem, "fail", str(e)


failed = []
skipped = ok = 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_warp_md_to_mni, p) for p in md_t1_files]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="MD→MNI"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

print(f"MNI MD: {len(list(ADC_MNI.glob('*_MD.nii.gz')))} | ok {ok} | skip {skipped} | failed {len(failed)}")
if failed:
    for name, err in failed[:5]:
        print(f"  {name}: {err}")

Apply T1 warps to 142 MD maps


MD→MNI: 100%|██████████| 142/142 [00:27<00:00,  5.25it/s]

MNI MD: 142 | ok 142 | skip 0 | failed 0


In [6]:
# --- Smooth MNI MD (5 mm FWHM), match long-MD preprocessing ------------------
fwhm_mm = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
mni_md = sorted(ADC_MNI.glob("*_MD.nii.gz"))
print(f"Smooth {len(mni_md)} MD volumes @ {fwhm_mm} mm FWHM")


def _smooth_md(pth: Path):
    adc_stem = pth.name.replace("_MD.nii.gz", "")
    out_path = ADC_SMOOTH / f"{adc_stem}_MD_s5.nii.gz"
    if out_path.exists():
        return adc_stem, "skip", ""
    try:
        img = ants.image_read(str(pth))
        sm = ants.smooth_image(
            img, sigma=sigma_mm, sigma_in_physical_coordinates=True,
        )
        # Scale mm2/s -> 10^-3 mm2/s (WM~0.7, GM~1.0, CSF~3.0)
        ants.image_write(sm * 1000.0, str(out_path))
        return adc_stem, "ok", ""
    except Exception as e:
        return adc_stem, "fail", str(e)


failed = []
skipped = ok = 0
max_workers = max(1, (os.cpu_count() or 4) - 2)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(_smooth_md, p) for p in mni_md]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Smooth MD"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

print(f"Smoothed MD: {len(list(ADC_SMOOTH.glob('*_MD_s5.nii.gz')))} | failed {len(failed)}")

Smooth 142 MD volumes @ 5.0 mm FWHM


Smooth MD: 100%|██████████| 142/142 [00:08<00:00, 16.13it/s]

Smoothed MD: 142 | failed 0


#### ML dataset preprocessing (tissue masks → parquets + `paired_df_short.csv`)

Dx from **DXSUM** (`DIAGNOSIS` 1/2/3). Amyloid from UC Berkeley spreadsheet.


In [5]:
# --- Resample MNI tissue masks to smoothed MD grid ----------------------------
import nilearn.image as nli

adc_smooth_files = sorted(ADC_SMOOTH.glob("*_MD_s5.nii.gz"))
if not adc_smooth_files:
    raise FileNotFoundError("No smoothed MD maps; run prior cells.")

ref_img = nib.load(adc_smooth_files[0])


def _load_mask(path: str):
    m = nli.resample_to_img(
        path, ref_img, interpolation="nearest", force_resample=True, copy_header=True,
    )
    return np.asarray(m.dataobj) > 0.5


gm_mask = _load_mask("model_data/mni_gm_mask.nii")
wm_mask = _load_mask("model_data/mni_wm_mask.nii")
csf_mask = _load_mask("model_data/mni_csf_mask.nii")
print(
    f"Ref shape {ref_img.shape} | GM {gm_mask.sum():,} | WM {wm_mask.sum():,} | CSF {csf_mask.sum():,}"
)

# --- Load subject pairings from paired_df_short_subjects.csv -----------------
_pairs_csv = Path("model_data/adni/paired_df_short_subjects.csv")
_pairs = pd.read_csv(_pairs_csv)

# Build image_subject_id keys (format used on disk: I<image_id>_<subject_id>)
_pairs["t1_image_subject_id"]  = _pairs["T1_Image_ID"]  + "_" + _pairs["T1_Subject"]
_pairs["dti_image_subject_id"] = _pairs["DTI_Image_ID"] + "_" + _pairs["DTI_Subject"]
_pairs["subject_id"] = _pairs["T1_Subject"]

# Keep only rows where both T1 and DTI files exist on disk
_t1_on_disk  = {p.name for p in T1_PVE_SMOOTH.iterdir() if p.is_dir()}
_dti_on_disk = {f.name.replace("_MD_s5.nii.gz", "") for f in ADC_SMOOTH.glob("*_MD_s5.nii.gz")}

paired_stems = _pairs[
    _pairs["t1_image_subject_id"].isin(_t1_on_disk) &
    _pairs["dti_image_subject_id"].isin(_dti_on_disk)
].reset_index(drop=True)

n_missing_t1  = (~_pairs["t1_image_subject_id"].isin(_t1_on_disk)).sum()
n_missing_dti = (~_pairs["dti_image_subject_id"].isin(_dti_on_disk)).sum()
print(f"CSV rows: {len(_pairs)} | paired (both on disk): {len(paired_stems)}")
if n_missing_t1:
    print(f"  Missing T1 ({n_missing_t1}):", list(_pairs.loc[~_pairs['t1_image_subject_id'].isin(_t1_on_disk), 't1_image_subject_id']))
if n_missing_dti:
    print(f"  Missing DTI ({n_missing_dti}):", list(_pairs.loc[~_pairs['dti_image_subject_id'].isin(_dti_on_disk), 'dti_image_subject_id']))

Ref shape (121, 145, 121) | GM 451,681 | WM 283,320 | CSF 173,822
CSV rows: 104 | paired (both on disk): 101
  Missing T1 (1): ['I820315_016_S_4952']
  Missing DTI (2): ['I1581804_114_S_6057', 'I1181934_153_S_6755']


In [6]:
# --- T1 parquets (paired rows only) -------------------------------------------
paired_t1_dirs = [T1_PVE_SMOOTH / s for s in paired_stems["t1_image_subject_id"]]


def _extract_t1(subj_dir: Path):
    stem = subj_dir.name
    csf = nib.load(subj_dir / f"{stem}_pve_0_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    gm  = nib.load(subj_dir / f"{stem}_pve_1_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    wm  = nib.load(subj_dir / f"{stem}_pve_2_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    return stem, gm[gm_mask], wm[wm_mask], csf[csf_mask]


stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, g, w, c in tqdm(
        ex.map(_extract_t1, paired_t1_dirs),
        total=len(paired_t1_dirs),
        desc="T1 parquets",
    ):
        stems.append(stem)
        gm_rows.append(g)
        wm_rows.append(w)
        csf_rows.append(c)

ix = pd.Index(stems, name="image_subject_id")
gm_df  = pd.DataFrame(np.vstack(gm_rows),  index=ix)
wm_df  = pd.DataFrame(np.vstack(wm_rows),  index=ix)
csf_df = pd.DataFrame(np.vstack(csf_rows), index=ix)
for df in (gm_df, wm_df, csf_df):
    df.columns = df.columns.astype(str)

gm_df.to_parquet(SHORT_T1  / "t1_short_masked_gm.parquet",  compression="zstd")
wm_df.to_parquet(SHORT_T1  / "t1_short_masked_wm.parquet",  compression="zstd")
csf_df.to_parquet(SHORT_T1 / "t1_short_masked_csf.parquet", compression="zstd")
print(gm_df.shape, wm_df.shape, csf_df.shape)

T1 parquets: 100%|██████████| 101/101 [00:01<00:00, 89.39it/s]


(101, 451681) (101, 283320) (101, 173822)


In [7]:
# --- DTI MD parquets (paired rows only) ---------------------------------------
def _extract_md(pth: Path):
    stem = pth.name.replace("_MD_s5.nii.gz", "")
    vol = nib.load(pth).get_fdata(dtype=np.float32)
    return stem, vol[gm_mask], vol[wm_mask], vol[csf_mask]


paired_dti_files = [ADC_SMOOTH / f"{s}_MD_s5.nii.gz" for s in paired_stems["dti_image_subject_id"]]
paired_existing  = [p for p in paired_dti_files if p.exists()]
if len(paired_existing) < len(paired_dti_files):
    print(f"Warning: {len(paired_dti_files) - len(paired_existing)} smoothed MD files missing")

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, g, w, c in tqdm(
        ex.map(_extract_md, paired_existing),
        total=len(paired_existing),
        desc="MD parquets",
    ):
        stems.append(stem)
        gm_rows.append(g)
        wm_rows.append(w)
        csf_rows.append(c)

ix = pd.Index(stems, name="image_subject_id")
adc_gm  = pd.DataFrame(np.vstack(gm_rows),  index=ix)
adc_wm  = pd.DataFrame(np.vstack(wm_rows),  index=ix)
adc_csf = pd.DataFrame(np.vstack(csf_rows), index=ix)
for df in (adc_gm, adc_wm, adc_csf):
    df.columns = df.columns.astype(str)

adc_gm.to_parquet(SHORT_DTI  / "dti_short_masked_gm_md.parquet",  compression="zstd")
adc_wm.to_parquet(SHORT_DTI  / "dti_short_masked_wm_md.parquet",  compression="zstd")
adc_csf.to_parquet(SHORT_DTI / "dti_short_masked_csf_md.parquet", compression="zstd")
print(adc_gm.shape, adc_wm.shape, adc_csf.shape)

MD parquets: 100%|██████████| 101/101 [00:00<00:00, 126.51it/s]


(101, 451681) (101, 283320) (101, 173822)


In [10]:
# --- paired_df_short.csv (Dx from DXSUM, not ADNIMERGE) ------------------------
meta_csv = Path("model_data/adni/paired_df_short.csv")

paired = paired_stems[["subject_id", "t1_image_subject_id", "dti_image_subject_id"]].copy()
paired["t1_image_id"]  = paired["t1_image_subject_id"].str.split("_").str[0].str.lstrip("I")
paired["dti_image_id"] = paired["dti_image_subject_id"].str.split("_").str[0].str.lstrip("I")

_dxsum_csv = Path("model_data/adni/DXSUM_22Sep2025.csv")
dxsum = pd.read_csv(
    _dxsum_csv,
    usecols=["PTID", "EXAMDATE", "DIAGNOSIS"],
    low_memory=False,
)
dxsum["EXAMDATE"]  = pd.to_datetime(dxsum["EXAMDATE"],  errors="coerce")
dxsum["DIAGNOSIS"] = pd.to_numeric(dxsum["DIAGNOSIS"],  errors="coerce")
dxsum = dxsum.dropna(subset=["DIAGNOSIS"]).loc[lambda z: z["DIAGNOSIS"].isin([1, 2, 3])]
dxsum = dxsum.sort_values("EXAMDATE").drop_duplicates("PTID", keep="last")
_dx_last = dxsum.set_index("PTID")["DIAGNOSIS"].astype(int)

paired["group"]      = paired["subject_id"].map(_dx_last.map({1: "CN", 2: "MCI", 3: "Dementia"}))
paired["diag_label"] = paired["subject_id"].map(_dx_last).map({1: 0, 2: 1, 3: 1})

amy = pd.read_csv(
    "model_data/adni/All_Subjects_UCBERKELEY_AMY_6MM_08Feb2026.csv",
    usecols=["PTID", "SCANDATE", "AMYLOID_STATUS"],
    low_memory=False,
)
amy = (
    amy.dropna(subset=["AMYLOID_STATUS"])
    .sort_values("SCANDATE")
    .drop_duplicates("PTID", keep="last")
)
paired["amyloid_label"] = paired["subject_id"].map(
    dict(zip(amy["PTID"], amy["AMYLOID_STATUS"]))
).astype("Int64")

paired = paired[[
    "subject_id",
    "t1_image_id",
    "dti_image_id",
    "t1_image_subject_id",
    "dti_image_subject_id",
    "group",
    "diag_label",
    "amyloid_label",
]]

# Keep only rows with a known amyloid status
paired = paired[paired["amyloid_label"].notna()].reset_index(drop=True)

paired.to_csv(meta_csv, index=False)
print(f"Wrote {meta_csv} ({len(paired)} rows with amyloid status)")
print(paired["group"].value_counts(dropna=False))
print(paired["amyloid_label"].value_counts(dropna=False))

Wrote model_data/adni/paired_df_short.csv (78 rows with amyloid status)
group
CN          47
MCI         21
Dementia    10
Name: count, dtype: int64
amyloid_label
0    47
1    31
Name: count, dtype: Int64


#### Combine short and long datasets

In [ ]:
# --- Combine short + long parquets → dti_t1_full_data -------------------------
SHORT_T1_DIR  = Path("model_data/adni/t1_short_data")
SHORT_DTI_DIR = Path("model_data/adni/dti_short_data")
LONG_T1_DIR   = Path("model_data/adni/t1_long_data")
LONG_DTI_DIR  = Path("model_data/adni/dti_long_data")
FULL_DIR      = Path("model_data/adni/dti_t1_full_data")

_short_meta = pd.read_csv("model_data/adni/paired_df_short.csv")
_short_t1_ids  = set(_short_meta["t1_image_subject_id"])
_short_dti_ids = set(_short_meta["dti_image_subject_id"])

_modalities = [
    # (short parquet,                   long parquet,                   t1_or_dti, output name)
    ("t1_short_masked_gm.parquet",      "t1_long_masked_gm.parquet",      "t1",  "full_masked_gm_t1.parquet"),
    ("t1_short_masked_wm.parquet",      "t1_long_masked_wm.parquet",      "t1",  "full_masked_wm_t1.parquet"),
    ("t1_short_masked_csf.parquet",     "t1_long_masked_csf.parquet",     "t1",  "full_masked_csf_t1.parquet"),
    ("dti_short_masked_gm_md.parquet",  "dti_long_masked_gm_md.parquet",  "dti", "full_masked_gm_dti.parquet"),
    ("dti_short_masked_wm_md.parquet",  "dti_long_masked_wm_md.parquet",  "dti", "full_masked_wm_dti.parquet"),
    ("dti_short_masked_csf_md.parquet", "dti_long_masked_csf_md.parquet", "dti", "full_masked_csf_dti.parquet"),
]

for short_name, long_name, modality, out_name in _modalities:
    short_dir = SHORT_T1_DIR  if modality == "t1" else SHORT_DTI_DIR
    long_dir  = LONG_T1_DIR   if modality == "t1" else LONG_DTI_DIR
    ids       = _short_t1_ids if modality == "t1" else _short_dti_ids

    short_df = pd.read_parquet(short_dir / short_name)
    long_df  = pd.read_parquet(long_dir  / long_name)

    short_filtered = short_df.loc[short_df.index.isin(ids)]

    combined = pd.concat([long_df, short_filtered])
    combined.to_parquet(FULL_DIR / out_name, compression="zstd")
    print(f"{out_name}: long {len(long_df)} + short {len(short_filtered)} = {len(combined)} rows")

full_masked_gm_t1.parquet: long 787 + short 78 = 865 rows
full_masked_wm_t1.parquet: long 787 + short 78 = 865 rows
full_masked_csf_t1.parquet: long 787 + short 78 = 865 rows
full_masked_gm_dti.parquet: long 787 + short 78 = 865 rows
full_masked_wm_dti.parquet: long 787 + short 78 = 865 rows
full_masked_csf_dti.parquet: long 787 + short 78 = 865 rows

paired_df_combined.csv: long 787 + short 78 = 865 rows
group
CN          468
MCI         307
Dementia     90
Name: count, dtype: int64
amyloid_label
1.0    407
0.0    395
NaN     63
Name: count, dtype: int64


In [15]:
# --- paired_df_combined.csv ---------------------------------------------------
_long_meta = pd.read_csv("model_data/adni/paired_df_long.csv")

# Keep only long rows with an amyloid label
_long_meta = _long_meta[_long_meta["amyloid_label"].notna()].reset_index(drop=True)

_combined_meta = pd.concat([_long_meta, _short_meta], ignore_index=True)
_combined_meta.to_csv(FULL_DIR / "paired_df_combined.csv", index=False)
print(f"\npaired_df_combined.csv: long {len(_long_meta)} + short {len(_short_meta)} = {len(_combined_meta)} rows")
print(_combined_meta["group"].value_counts(dropna=False))
print(_combined_meta["amyloid_label"].value_counts(dropna=False))


paired_df_combined.csv: long 724 + short 78 = 802 rows
group
CN          434
MCI         286
Dementia     82
Name: count, dtype: int64
amyloid_label
1.0    407
0.0    395
Name: count, dtype: int64
